[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShintaroMinami/notebook_proteina_complexa/blob/main/proteina_complexa.ipynb)

# Proteina Complexa — Hands-On Notebook

このノートブックでは、NVIDIAが開発したタンパク質デザインモデル **Proteina Complexa** を
Google Colab上で実行します。

> **必要なもの:** Googleアカウントのみ。ローカル環境の構築は不要です。

---
## 1. 環境構築

はじめに、必要なソフトウェアをインストールします。
以下のセルを **上から順に** 実行してください（各セルで Shift+Enter）。

### 1.1 GPU の確認

まず、GPUが割り当てられているか確認します。
「GPU が見つかりません」と表示された場合は、メニューの **「ランタイム」→「ランタイムのタイプを変更」** から
**GPU** を選択してください。

In [ ]:
import subprocess, sys

result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                        capture_output=True, text=True)
if result.returncode == 0:
    gpu_info = result.stdout.strip()
    print(f"GPU: {gpu_info}")
else:
    print("GPU が見つかりません。ランタイムのタイプを変更してください。")

print(f"Python: {sys.version}")

### 1.2 パッケージマネージャ (uv) のインストールとリポジトリの取得

高速パッケージマネージャ **uv** をインストールし、Proteina Complexa のソースコードを取得します。

In [ ]:
!pip install -q uv
!git clone --depth 1 https://github.com/NVIDIA-Digital-Bio/Proteina-Complexa.git
%cd Proteina-Complexa
print("Done.")

### 1.3 PyTorch のインストール

GPU対応版の PyTorch をインストールします。（数分かかります）

In [ ]:
!uv pip install --system torch==2.7.0+cu126 torchvision torchaudio \
    --index-url https://download.pytorch.org/whl/cu126

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### 1.4 Proteina Complexa のインストール

本体と依存ライブラリをインストールします。（数分かかります）

In [ ]:
# Install base package
!uv pip install --system --index-strategy unsafe-best-match -e .

# Install PyTorch Geometric
!uv pip install --system torch_geometric torch_scatter torch_sparse torch_cluster \
    -f https://data.pyg.org/whl/torch-2.7.0+cu126.html

# Install Graphein and Atomworks
!uv pip install --system graphein==1.7.7 --no-deps
!uv pip install --system "atomworks[ml,openbabel,dev]" 2>/dev/null || echo "Warning: atomworks partial install"

# Install biotite
!uv pip install --system biotite==1.6.0

print("Done.")

### 1.5 追加ライブラリのインストール（評価用）

ColabDesign / JAX など、構造予測・評価に使うライブラリをインストールします。

In [ ]:
# ColabDesign & AlphaFold-ColabFold
!uv pip install --system colabdesign==1.1.1 alphafold-colabfold==2.3.7
!uv pip install --system -e community_models/colabdesign

# JAX with CUDA
!uv pip install --system jaxlib==0.4.29+cuda12.cudnn91 \
    -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
!uv pip install --system "jax[cuda12]==0.4.29" \
    -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
!uv pip install --system flax==0.9.0 --no-deps

# tmol
!uv pip install --system "llvmlite>=0.41" "numba>=0.59" 2>/dev/null || true
!uv pip install --system "git+https://github.com/uw-ipd/tmol.git@d8a6f7f9649d36e74440bca25246ee7c467ce490" 2>/dev/null || echo "Warning: tmol install failed (non-critical)"

# Foundry
!uv pip install --system "rc-foundry[all]" 2>/dev/null || echo "Warning: rc-foundry install failed (non-critical)"

print("Done.")

### 1.6 モデルの重みをダウンロード

事前学習済みモデルをダウンロードします。（数分かかります）

In [ ]:
!complexa init
!complexa download --complexa-all

### 1.7 インストールの確認

全てのインストールが完了したか確認します。

In [ ]:
import torch
import proteinfoundation

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Proteina Complexa: installed")
print()
print("Environment setup complete!")